# RideWise -- Data Preprocessing

Fixing all data quality issues identified during our EDA prepare clean datset fot feature engineering & Modelling

In [1]:
### Setup & import libraries

import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


warnings.filterwarnings("ignore")
%matplotlib inline
sns.set_theme(style="whitegrid", font_scale=1.2)


## define the location of our raw and processed 
DATA_RAW = os.path.join("..", "data", "raw")
DATA_PROCESSED = os.path.join("..", "data", "processed")
os.makedirs(DATA_PROCESSED, exist_ok=True)

In [2]:
# Loading the datasets
riders = pd.read_csv(os.path.join(DATA_RAW, "riders.csv"), parse_dates=["signup_date"])
trips = pd.read_csv(os.path.join(DATA_RAW, "trips.csv"))
drivers = pd.read_csv(os.path.join(DATA_RAW, "drivers.csv"), parse_dates=["signup_date"])
sessions = pd.read_csv(os.path.join(DATA_RAW, "sessions.csv"))
promotions = pd.read_csv(os.path.join(DATA_RAW, "promotions.csv"))

In [3]:
# Create  working copies

riders_clean = riders.copy()
trips_clean = trips.copy()
drivers_clean = drivers.copy()
sessions_clean = sessions.copy()
promotions_clean = promotions.copy()

print("Working copies created.")

Working copies created.


### STEP 1 — Initial Dataset overview

- Before cleaning the data, we inspect the size and structure of every dataset.

- This provides a baseline that can later be compared with the cleaned versions.

In [4]:
datasets = {
    "riders": riders_clean,
    "trips": trips_clean,
    "drivers": drivers_clean,
    "sessions": sessions_clean,
    "promotions": promotions_clean
}

overview = []

for name, df in datasets.items():
    overview.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicate_rows": df.duplicated().sum(),
        "missing_values": df.isna().sum().sum()
    })

overview_df = pd.DataFrame(overview)

overview_df

,dataset,rows,columns,duplicate_rows,missing_values
0,riders,10000,8,0,6947
1,trips,200000,16,0,0
2,drivers,5000,7,0,0
3,sessions,50000,8,0,0
4,promotions,20,11,0,0


### STEP 2 - Data Inspection and Quality Checks

In [5]:
# Inspect columns and dataset

for name, df in datasets.items():
    print("=" * 70)
    print(f"{name.upper()} DATASET")
    print("=" * 70)

    print("\nShape:")
    print(df.shape)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nData types:")
    print(df.dtypes)

    print()

RIDERS DATASET

Shape:
(10000, 8)

Columns:
['user_id', 'signup_date', 'loyalty_status', 'age', 'city', 'avg_rating_given', 'churn_prob', 'referred_by']

Data types:
user_id                        str
signup_date         datetime64[us]
loyalty_status                 str
age                        float64
city                           str
avg_rating_given           float64
churn_prob                 float64
referred_by                    str
dtype: object

TRIPS DATASET

Shape:
(200000, 16)

Columns:
['trip_id', 'user_id', 'driver_id', 'fare', 'surge_multiplier', 'tip', 'payment_type', 'pickup_time', 'dropoff_time', 'pickup_lat', 'pickup_lng', 'dropoff_lat', 'dropoff_lng', 'weather', 'city', 'loyalty_status']

Data types:
trip_id                 str
user_id                 str
driver_id               str
fare                float64
surge_multiplier    float64
tip                 float64
payment_type            str
pickup_time             str
dropoff_time            str
pickup_lat      

In [6]:
# Missing value report

def missing_value_report(df):
    """
    Return the number and percentage of missing values
    for columns that contain at least one missing value.
    """

    report = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percentage": df.isna().mean().mul(100)
    })

    report = report[report["missing_count"] > 0]

    return report.sort_values(
        by="missing_percentage",
        ascending=False
    ).round(2)

In [7]:
for name, df in datasets.items():
    print("=" * 70)
    print(f"{name.upper()} — MISSING VALUES")
    print("=" * 70)

    report = missing_value_report(df)

    if report.empty:
        print("No missing values found.")
    else:
        display(report)

    print()

RIDERS — MISSING VALUES


,missing_count,missing_percentage
referred_by,6947,69.47



TRIPS — MISSING VALUES
No missing values found.

DRIVERS — MISSING VALUES
No missing values found.

SESSIONS — MISSING VALUES
No missing values found.

PROMOTIONS — MISSING VALUES
No missing values found.



In [8]:
# Step  — Validate unique identifiers

identifier_checks = {
    "riders": "user_id",
    "trips": "trip_id",
    "drivers": "driver_id",
    "sessions": "session_id"
}

for dataset_name, id_column in identifier_checks.items():

    df = datasets[dataset_name]

    duplicate_ids = df[id_column].duplicated().sum()

    print(
        f"{dataset_name:<10} | "
        f"ID column: {id_column:<12} | "
        f"Duplicate IDs: {duplicate_ids:,}"
    )

riders     | ID column: user_id      | Duplicate IDs: 0
trips      | ID column: trip_id      | Duplicate IDs: 0
drivers    | ID column: driver_id    | Duplicate IDs: 0
sessions   | ID column: session_id   | Duplicate IDs: 0


In [9]:
# Step — Inspect date ranges

# Churn prediction is a time-dependent problem.

# Before defining observation and prediction windows, we must understand:

# when rider accounts were created;
# the first and last trip dates;
# the session activity period;
# whether the available data provides enough future time to measure churn.

# Convert timestamp columns consistently

for col in ["pickup_time", "dropoff_time"]:
    trips_clean[col] = (
        pd.to_datetime(
            trips_clean[col],
            utc=True,
            errors="coerce"
        )
        .dt.tz_localize(None)
    )


sessions_clean["session_time"] = (
    pd.to_datetime(
        sessions_clean["session_time"],
        utc=True,
        errors="coerce"
    )
    .dt.tz_localize(None)
)


drivers_clean["last_active"] = (
    pd.to_datetime(
        drivers_clean["last_active"],
        errors="coerce"
    )
)

In [10]:
# check available timeline
# determine if 30-day, 60-day or 90-day prediction window is realistic.

date_ranges = pd.DataFrame({
    "dataset": [
        "riders",
        "trips",
        "sessions",
        "drivers"
    ],

    "date_column": [
        "signup_date",
        "pickup_time",
        "session_time",
        "last_active"
    ],

    "minimum_date": [
        riders_clean["signup_date"].min(),
        trips_clean["pickup_time"].min(),
        sessions_clean["session_time"].min(),
        drivers_clean["last_active"].min()
    ],

    "maximum_date": [
        riders_clean["signup_date"].max(),
        trips_clean["pickup_time"].max(),
        sessions_clean["session_time"].max(),
        drivers_clean["last_active"].max()
    ]
})

date_ranges["available_days"] = (
    date_ranges["maximum_date"] -
    date_ranges["minimum_date"]
).dt.days

date_ranges

,dataset,date_column,minimum_date,maximum_date,available_days
0,riders,signup_date,2023-04-27 00:00:00.000000,2025-04-26 00:00:00.000000,730
1,trips,pickup_time,2024-04-26 21:40:34.000000,2025-04-27 23:43:26.000000,366
2,sessions,session_time,2025-04-26 21:33:06.000000,2025-04-27 23:45:59.000000,1
3,drivers,last_active,2022-04-28 16:45:23.765979,2025-04-27 16:20:56.061019,1094


### Check the original churn variables

- The riders table contains a `churn_prob` variable.

- This variable will be inspected for data understanding, but it will not be converted into the final churn target because:

1. it is not an observed churn outcome;
2. its calculation method is unknown;
3. converting it at 0.5 creates an artificial target;
4. it may not align with the rider behaviour contained in the trips and sessions ta

In [11]:
if "churn_prob" in riders_clean.columns:

    print(
        riders_clean["churn_prob"]
        .describe(
            percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]
        )
        .round(3)
    )

    print("\nValues outside the expected 0–1 range:")

    invalid_churn_prob = (
        (riders_clean["churn_prob"] < 0) |
        (riders_clean["churn_prob"] > 1)
    ).sum()

    print(invalid_churn_prob)

count    10000.000
mean         0.286
std          0.159
min          0.003
1%           0.027
25%          0.162
50%          0.267
75%          0.389
99%          0.699
max          0.913
Name: churn_prob, dtype: float64

Values outside the expected 0–1 range:
0


### clean the Riders Dataset

In [15]:
# Validate Age

# Age should be a whole number.

# Since age is recorded as a decimal, we'll round it to the nearest whole year.

# Check the age distribution

riders_clean["age"].describe().round(2)

# Convert age to whole years

riders_clean["age"] = (
    riders_clean["age"]
    .round()
    .astype(int)
)
# Validate the result

print(riders_clean["age"].describe())

count    10000.000000
mean        35.156000
std          9.549586
min         18.000000
25%         28.000000
50%         35.000000
75%         42.000000
max         70.000000
Name: age, dtype: float64


In [ ]:
# step validate Age Range

print(
    riders_clean["age"].between(18, 100).value_counts()
)

age
True    10000
Name: count, dtype: int64


In [18]:
# step  Create referral indicator
# we want to generate a binary referal indicator, was the rider referred?  Yes / No
# instead of treating not referred as a missing values. This way we preserve the behavioural signal

riders_clean["was_referred"] = (
    riders_clean["referred_by"]
    .notna()
    .astype(int)
)

In [ ]:
# verify the column
riders_clean["was_referred"].value_counts()

was_referred
0    6947
1    3053
Name: count, dtype: int64

In [21]:
# drop original column.
riders_clean = riders_clean.drop(columns=["referred_by"])

In [22]:
# step Final Validation
print(riders_clean.info())
riders_clean.head()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   user_id           10000 non-null  str           
 1   signup_date       10000 non-null  datetime64[us]
 2   loyalty_status    10000 non-null  str           
 3   age               10000 non-null  int64         
 4   city              10000 non-null  str           
 5   avg_rating_given  10000 non-null  float64       
 6   churn_prob        10000 non-null  float64       
 7   was_referred      10000 non-null  int64         
dtypes: datetime64[us](1), float64(2), int64(2), str(3)
memory usage: 796.5 KB
None


,user_id,signup_date,loyalty_status,age,city,avg_rating_given,churn_prob,was_referred
0,R00000,2025-01-24,Bronze,35,Nairobi,5.0,0.142431,1
1,R00001,2024-09-09,Bronze,35,Nairobi,4.7,0.674161,0
2,R00002,2024-09-07,Bronze,47,Lagos,4.2,0.510379,0
3,R00003,2025-03-17,Bronze,42,Nairobi,4.9,0.244779,0
4,R00004,2024-08-20,Silver,41,Lagos,3.9,0.269960,1


### STEP - Clean Trips Dataset

In [23]:
# step - Remove Invalid Dates

# A trip cannot exist without both:

# pickup time
# drop-off time

before_rows = len(trips_clean)

trips_clean = trips_clean.dropna(
    subset=["pickup_time", "dropoff_time"]
)

after_rows = len(trips_clean)

print(f"Rows removed: {before_rows - after_rows:,}")

Rows removed: 0


In [24]:
# Calculate The Trip Duration

trips_clean["trip_duration_minutes"] = (
    trips_clean["dropoff_time"] -
    trips_clean["pickup_time"]
).dt.total_seconds() / 60

In [25]:
# Validate the trip duration
trips_clean["trip_duration_minutes"].describe().round(2)

count    200000.00
mean         31.96
std          15.87
min           5.00
25%          18.00
50%          32.00
75%          46.00
max          59.00
Name: trip_duration_minutes, dtype: float64

In [26]:
# Step  – Remove Impossible Trips

# A ride cannot last:

# negative minutes
# zero minutes


before_rows = len(trips_clean)

trips_clean = trips_clean[
    trips_clean["trip_duration_minutes"] > 0
]

after_rows = len(trips_clean)

print(f"Trips removed: {before_rows - after_rows:,}")

Trips removed: 0


In [27]:
# validate fares

print(trips_clean["fare"].describe().round(2))

print(
    "\nNegative fares:",
    (trips_clean["fare"] < 0).sum()
)

count    200000.00
mean         15.40
std           6.16
min           2.97
25%          11.00
50%          14.13
75%          18.35
max          82.74
Name: fare, dtype: float64

Negative fares: 0


In [28]:
# Validate surge Multipliers

print(trips_clean["surge_multiplier"].describe())

print(
    "\nValues below 1:",
    (trips_clean["surge_multiplier"] < 1).sum()
)

count    200000.000000
mean          1.141500
std           0.255362
min           1.000000
25%           1.000000
50%           1.000000
75%           1.200000
max           3.800000
Name: surge_multiplier, dtype: float64

Values below 1: 0


In [29]:
# Validate geographical cordinates

coordinate_checks = {

    "pickup_lat":
        trips_clean["pickup_lat"].between(-90, 90),

    "dropoff_lat":
        trips_clean["dropoff_lat"].between(-90, 90),

    "pickup_lng":
        trips_clean["pickup_lng"].between(-180, 180),

    "dropoff_lng":
        trips_clean["dropoff_lng"].between(-180, 180)

}

for col, check in coordinate_checks.items():

    print(
        f"{col:<15}: Invalid values = {(~check).sum()}"
    )

pickup_lat     : Invalid values = 0
dropoff_lat    : Invalid values = 0
pickup_lng     : Invalid values = 0
dropoff_lng    : Invalid values = 0


In [30]:
# Final Trip Validation
print(trips_clean.info())
trips_clean.head()

<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 17 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   trip_id                200000 non-null  str           
 1   user_id                200000 non-null  str           
 2   driver_id              200000 non-null  str           
 3   fare                   200000 non-null  float64       
 4   surge_multiplier       200000 non-null  float64       
 5   tip                    200000 non-null  float64       
 6   payment_type           200000 non-null  str           
 7   pickup_time            200000 non-null  datetime64[us]
 8   dropoff_time           200000 non-null  datetime64[us]
 9   pickup_lat             200000 non-null  float64       
 10  pickup_lng             200000 non-null  float64       
 11  dropoff_lat            200000 non-null  float64       
 12  dropoff_lng            200000 non-null  float64       


,trip_id,user_id,driver_id,fare,surge_multiplier,tip,payment_type,pickup_time,dropoff_time,pickup_lat,pickup_lng,dropoff_lat,dropoff_lng,weather,city,loyalty_status,trip_duration_minutes
0,T000000,R05207,D00315,12.11,1.0,0.00,Card,2024-11-27 16:14:50,2024-11-27 17:06:50,-1.108123,36.912209,-1.068155,36.875377,Foggy,Nairobi,Bronze,52.0
1,T000001,R09453,D03717,8.73,1.0,0.02,Card,2024-10-28 22:59:48,2024-10-28 23:12:48,6.675266,3.515740,6.641734,3.525620,Sunny,Lagos,Gold,13.0
2,T000002,R00567,D02035,19.68,1.0,0.00,Card,2025-02-17 03:09:41,2025-02-17 03:25:41,-1.248589,37.010668,-1.273182,37.018586,Cloudy,Nairobi,Bronze,16.0
3,T000003,R09573,D02657,16.43,1.0,0.01,Mobile Money,2024-06-18 17:22:14,2024-06-18 17:27:14,29.819554,31.188780,29.837689,31.232978,Cloudy,Cairo,Bronze,5.0
4,T000004,R03446,D01026,8.70,1.0,1.06,Card,2024-10-05 07:31:16,2024-10-05 08:01:16,-1.676479,36.729219,-1.638395,36.694063,Sunny,Nairobi,Gold,30.0


### STEP - Clean Drivers Dataset

In [ ]:
# Inspect the Dataset
print(drivers_clean.info())
print(drivers_clean.head())

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   driver_id        5000 non-null   str           
 1   rating           5000 non-null   float64       
 2   vehicle_type     5000 non-null   str           
 3   signup_date      5000 non-null   datetime64[us]
 4   last_active      5000 non-null   datetime64[us]
 5   city             5000 non-null   str           
 6   acceptance_rate  5000 non-null   float64       
dtypes: datetime64[us](2), float64(2), str(3)
memory usage: 355.1 KB
None


,driver_id,rating,vehicle_type,signup_date,last_active,city,acceptance_rate
0,D00000,3.1,SUV,2025-01-20,2025-01-06 18:23:09.312275,Cairo,0.679555
1,D00001,5.0,Sedan,2023-03-27,2025-04-27 01:44:02.472554,Nairobi,0.548786
2,D00002,4.5,Motorcycle,2024-05-02,2025-03-07 19:24:46.367672,Nairobi,0.593724
3,D00003,5.0,Motorcycle,2023-04-16,2025-03-26 19:16:24.253793,Nairobi,0.990000
4,D00004,4.4,Motorcycle,2023-05-28,2025-04-08 18:54:44.649615,Lagos,0.519773


In [32]:
# check driver duplicate

duplicate_drivers = (
    drivers_clean["driver_id"]
    .duplicated()
    .sum()
)

print(f"Duplicate driver IDs: {duplicate_drivers}")

Duplicate driver IDs: 0


In [33]:
# Validate Driver Ratings

# Most ride-hailing platforms use ratings between 1 and 5.

drivers_clean["rating"].describe().round(2)



count    5000.00
mean        4.17
std         0.59
min         3.10
25%         3.70
50%         4.20
75%         4.70
max         5.00
Name: rating, dtype: float64

In [34]:
# Check for invalid ratings:

invalid_rating = (
    ~drivers_clean["rating"].between(1, 5)
).sum()

print(f"Invalid ratings: {invalid_rating}")

Invalid ratings: 0


In [35]:
# Validate Acceptance Rate

# Acceptance rate is a percentage and should lie between 0 and 100.

drivers_clean["acceptance_rate"].describe().round(2)


count    5000.00
mean        0.70
std         0.19
min         0.10
25%         0.57
50%         0.70
75%         0.84
max         0.99
Name: acceptance_rate, dtype: float64

In [39]:
invalid_acceptance = (
    ~drivers_clean["acceptance_rate"].between(0, 100)
).sum()

print(
    f"Invalid acceptance rates: {invalid_acceptance}"
)

Invalid acceptance rates: 0


In [42]:
# Missing Value Assessment
missing_driver = (
    drivers_clean
    .isna()
    .sum()
    .to_frame("Missing")
)

missing_driver["Percentage"] = (
    missing_driver["Missing"]
    / len(drivers_clean)
    * 100
).round(2)

missing_driver[
    missing_driver["Missing"] > 0
]

,Missing,Percentage


In [43]:
# Final Validation
print(drivers_clean.info())
print(drivers_clean.head())

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   driver_id        5000 non-null   str           
 1   rating           5000 non-null   float64       
 2   vehicle_type     5000 non-null   str           
 3   signup_date      5000 non-null   datetime64[us]
 4   last_active      5000 non-null   datetime64[us]
 5   city             5000 non-null   str           
 6   acceptance_rate  5000 non-null   float64       
dtypes: datetime64[us](2), float64(2), str(3)
memory usage: 355.1 KB
None
  driver_id  rating vehicle_type signup_date                last_active  \
0    D00000     3.1          SUV  2025-01-20 2025-01-06 18:23:09.312275   
1    D00001     5.0        Sedan  2023-03-27 2025-04-27 01:44:02.472554   
2    D00002     4.5   Motorcycle  2024-05-02 2025-03-07 19:24:46.367672   
3    D00003     5.0   Motorcycle  2023-04-16 2025-03-2

### Clean Session Dataset

- The sessions dataset records how riders interact with the RideWise mobile application before booking trips.

- These interactions provide valuable behavioural information, such as app engagement and browsing activity. However, before using the data for feature 

engineering, we must ensure it is accurate and complete.

In [44]:
# Inspect the Dataset
print(sessions_clean.info())
print(sessions_clean.head())

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   session_id      50000 non-null  str           
 1   rider_id        50000 non-null  str           
 2   session_time    50000 non-null  datetime64[us]
 3   time_on_app     50000 non-null  int64         
 4   pages_visited   50000 non-null  int64         
 5   converted       50000 non-null  int64         
 6   city            50000 non-null  str           
 7   loyalty_status  50000 non-null  str           
dtypes: datetime64[us](1), int64(3), str(4)
memory usage: 4.2 MB
None
  session_id rider_id        session_time  time_on_app  pages_visited  \
0    S000000   R08605 2025-04-27 16:52:06           79              4   
1    S000001   R08823 2025-04-27 05:05:22          101              3   
2    S000002   R05342 2025-04-27 21:12:25           12              1   
3    S000003   R05057

In [45]:
# Check Duplicate Sessions

# Each session should have a unique session ID.

duplicate_sessions = (
    sessions_clean["session_id"]
    .duplicated()
    .sum()
)

print(f"Duplicate session IDs: {duplicate_sessions}")

Duplicate session IDs: 0


In [46]:
# Validate App Usage Time

# App usage time cannot be negative.

sessions_clean["time_on_app"].describe().round(2)


count    50000.00
mean        97.94
std        211.68
min          0.00
25%         12.00
50%         35.00
75%         90.00
max       1800.00
Name: time_on_app, dtype: float64

In [51]:
invalid_time = (
     sessions_clean["time_on_app"] < 0
).sum()

print(f"Negative app times: {invalid_time}")

Negative app times: 0


In [52]:
# Validate Pages Visited

# A rider cannot visit a negative number of pages.

sessions_clean["pages_visited"].describe().round(2)

count    50000.00
mean         2.77
std          1.55
min          1.00
25%          1.00
50%          2.00
75%          4.00
max          5.00
Name: pages_visited, dtype: float64

In [53]:
invalid_pages = (
    sessions_clean["pages_visited"] < 0
).sum()

print(
    f"Negative pages visited: {invalid_pages}"
)

Negative pages visited: 0


In [61]:
# Missing Value Assessment

# Rather than filling missing values immediately, we'll inspect them.

missing_sessions = (
    sessions_clean
    .isna()
    .sum()
    .to_frame("Missing")
)

missing_sessions["Percentage"] = (
    missing_sessions["Missing"]
    / len(sessions_clean)
    * 100
).round(2)

missing_sessions[
    missing_sessions["Missing"] > 0
]



,Missing,Percentage


In [62]:
# Final Validation
print(sessions_clean.info())
print(sessions_clean.head())

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   session_id      50000 non-null  str           
 1   rider_id        50000 non-null  str           
 2   session_time    50000 non-null  datetime64[us]
 3   time_on_app     50000 non-null  int64         
 4   pages_visited   50000 non-null  int64         
 5   converted       50000 non-null  int64         
 6   city            50000 non-null  str           
 7   loyalty_status  50000 non-null  str           
dtypes: datetime64[us](1), int64(3), str(4)
memory usage: 4.2 MB
None
  session_id rider_id        session_time  time_on_app  pages_visited  \
0    S000000   R08605 2025-04-27 16:52:06           79              4   
1    S000001   R08823 2025-04-27 05:05:22          101              3   
2    S000002   R05342 2025-04-27 21:12:25           12              1   
3    S000003   R05057

### Final validation and Save Clean Data

In [64]:
# Confirm Clean Dataset Shapes
print("Clean dataset shapes")
print("-" * 35)

print(f"Riders   : {riders_clean.shape}")
print(f"Trips    : {trips_clean.shape}")
print(f"Drivers  : {drivers_clean.shape}")
print(f"Sessions : {sessions_clean.shape}")

Clean dataset shapes
-----------------------------------
Riders   : (10000, 8)
Trips    : (200000, 17)
Drivers  : (5000, 7)
Sessions : (50000, 8)


In [77]:
# primary key validation

primary_key_checks = {
    "riders_clean": (
        riders_clean,
        "user_id"
    ),
    "trips_clean": (
        trips_clean,
        "trip_id"
    ),
    "drivers_clean": (
        drivers_clean,
        "driver_id"
    ),
    "sessions_clean": (
        sessions_clean,
        "session_id"
    )
}

print("Primary key validation")
print("-" * 35)

for dataset_name, (df, key_col) in primary_key_checks.items():

    duplicate_count = df[key_col].duplicated().sum()

    print(
        f"{dataset_name:<18} "
        f"{key_col:<12} "
        f"duplicates: {duplicate_count}"
    )

Primary key validation
-----------------------------------
riders_clean       user_id      duplicates: 0
trips_clean        trip_id      duplicates: 0
drivers_clean      driver_id    duplicates: 0
sessions_clean     session_id   duplicates: 0


In [78]:
# Valid rider IDs from the riders table

valid_rider_ids = set(riders_clean["user_id"])

# Check rider IDs in trips

invalid_trip_riders = (
    ~trips_clean["user_id"].isin(valid_rider_ids)
).sum()

# Sessions uses rider_id, not user_id

invalid_session_riders = (
    ~sessions_clean["rider_id"].isin(valid_rider_ids)
).sum()

print("Rider referential integrity")
print("-" * 40)

print(
    f"Trip records with unknown riders    : "
    f"{invalid_trip_riders}"
)

print(
    f"Session records with unknown riders : "
    f"{invalid_session_riders}"
)

Rider referential integrity
----------------------------------------
Trip records with unknown riders    : 0
Session records with unknown riders : 0


In [72]:
# Check Driver Referential Integrity

# Every driver referenced in the trips table should exist in the drivers table.

valid_driver_ids = set(drivers_clean["driver_id"])

invalid_trip_drivers = (
    ~trips_clean["driver_id"].isin(valid_driver_ids)
).sum()

print("Driver referential integrity")
print("-" * 35)

print(
    f"Trip records with unknown drivers: "
    f"{invalid_trip_drivers}"
)

Driver referential integrity
-----------------------------------
Trip records with unknown drivers: 0


In [79]:
# Save the Clean Datasets

riders_clean.to_csv(
    os.path.join(
        DATA_PROCESSED,
        "riders_clean.csv"
    ),
    index=False
)

trips_clean.to_csv(
    os.path.join(
        DATA_PROCESSED,
        "trips_clean.csv"
    ),
    index=False
)

drivers_clean.to_csv(
    os.path.join(
        DATA_PROCESSED,
        "drivers_clean.csv"
    ),
    index=False
)

sessions_clean.to_csv(
    os.path.join(
        DATA_PROCESSED,
        "sessions_clean.csv"
    ),
    index=False
)

print("Clean datasets saved successfully.")

Clean datasets saved successfully.
